In [1]:
!pip install -q transformers peft bitsandbytes accelerate


[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import torch
from huggingface_hub import login

# 1. Local Paths - UPDATE THESE to your actual local folders
# Example: "C:/Users/Alina/Models/hikelogic-lora-adapter"
BASE_MODEL = "OpenLLM-Ro/RoMistral-7b-Instruct"
LORA_PATH = "./hikelogic-lora-adapter-mistral-7B"

# 2. Check for GPU (Crucial for local runs)
if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("WARNING: No GPU detected. Running a 7B model on CPU will be extremely slow.")

# 3. Login (Optional - only needed if the base model is gated or you want to push)
# login(token="your_hf_token_here")

c:\Users\Alina\Desktop\UDLLM\hike-logic-app\HikeLogic\hikelogic-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import torch
print(torch.__version__)

2.4.1+cu121


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel

print("Loading base model in 4-bit...")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

print("Applying LoRA adapter...")
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval() # Set to evaluation mode

# Create the generation pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("HikeLogic is ready!")

Loading base model in 4-bit...


Loading weights: 100%|██████████| 291/291 [07:04<00:00,  1.46s/it]


Applying LoRA adapter...


ValueError: Can't find 'adapter_config.json' at './hikelogic-lora-adapter-mistral-7B'

In [ ]:
def ask_hikelogic(question, context):
    system_prompt = "You are HikeLogic, an assistant for Romanian hiking trails. Answer using ONLY the context provided."
    
    # Matching your fine-tuning format
    prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\nTrail context:\n{context}\n\nQuestion: {question}<|im_end|>\n<|im_start|>assistant\n"
    
    outputs = pipe(
        prompt, 
        max_new_tokens=256, 
        do_sample=True, 
        temperature=0.1, 
        top_k=50, 
        top_p=0.95
    )
    
    # Cleaning up the output to show only the answer
    generated_text = outputs[0]["generated_text"]
    answer = generated_text.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
    return answer

# --- TEST ---
context_test = "[1] Traseu: Bușteni - Cascada Urlătoarea. Marcaj: punct roșu. Durată: 1.5 ore. Dificultate: ușor."
question_test = "Ce marcaj are traseul către Cascada Urlătoarea?"

print(f"Răspuns: {ask_hikelogic(question_test, context_test)}")